In [ ]:
# =========================================================================
# 1. 環境設定與資料集下載
# =========================================================================
import os

# 下載並解壓縮動漫頭像資料集
if not os.path.exists("./faces"):
    print("[INFO] 正在下載與解壓縮資料集...")
    # 在 ipynb 檔裡面，前面加上驚嘆號 ! 就可以直接執行 Linux 系統命令
    !curl -s https://packagecloud.io/install/repositories/github/git-lfs/script.deb.sh | bash
    !apt-get install git-lfs
    !git lfs install
    !git clone https://huggingface.co/datasets/LeoFeng/MLHW_6
    !unzip -q ./MLHW_6/faces.zip -d .
    print(" -> [SUCCESS] 資料集解壓完成。")
else:
    print("[INFO] 資料集已存在，跳過下載。")

In [ ]:
# =========================================================================
# 2. 導入套件與資料集預處理
# =========================================================================
import glob
import random
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch import optim
from torch.utils.data import Dataset, DataLoader
import torchvision.utils as vutils
from IPython.display import display, clear_output  # 💡 專門用來在網頁裡動態重新整理圖片的工具

def same_seeds(seed):
    """固定隨機種子，確保每次實驗結果的可重複性"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(2022)
workspace_dir = '.'

class CrypkoDataset(Dataset):
    def __init__(self, fnames, transform):
        self.transform = transform
        self.fnames = fnames
        self.num_samples = len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img = torchvision.io.read_image(fname)
        img = self.transform(img)
        return img

    def __len__(self):
        return self.num_samples

def get_dataset(root):
    fnames = glob.glob(os.path.join(root, '*'))
    compose = [
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
    transform = transforms.Compose(compose)
    return CrypkoDataset(fnames, transform)

print("[INFO] 資料處理函數與管理員已準備就緒！")

In [ ]:
# =========================================================================
# 3. 模型架構定義 (Model Architecture)
# =========================================================================
def weights_init(m):
    """權重初始化"""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        m.weight.data.normal_(0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)

class Generator(nn.Module):
    def __init__(self, in_dim, dim=64):
        super(Generator, self).__init__()
        def GAN_Block(in_feat, out_feat, normalize=True):
            layers = [nn.ConvTranspose2d(in_feat, out_feat, 4, stride=2, padding=1, bias=False)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_feat))
            layers.append(nn.ReLU())
            return layers

        self.l1 = nn.Sequential(
            nn.ConvTranspose2d(in_dim, dim * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(dim * 8),
            nn.ReLU()
        )
        self.l2 = nn.Sequential(*GAN_Block(dim * 8, dim * 4))
        self.l3 = nn.Sequential(*GAN_Block(dim * 4, dim * 2))
        self.l4 = nn.Sequential(*GAN_Block(dim * 2, dim))
        self.l5 = nn.Sequential(
            nn.ConvTranspose2d(dim, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )
        self.apply(weights_init) # 💡 傳遞函式指標巡房初始化

    def forward(self, x):
        return self.l5(self.l4(self.l3(self.l2(self.l1(x)))))


class Discriminator(nn.Module):
    def __init__(self, in_dim, dim=64):
        super(Discriminator, self).__init__()
        def GAN_Block(in_feat, out_feat, normalize=True):
            layers = [nn.Conv2d(in_feat, out_feat, 4, stride=2, padding=1, bias=False)]
            if normalize:
                layers.append(nn.LayerNorm([out_feat, 1, 1]))
            layers.append(nn.LeakyReLU(0.2))
            return layers

        self.l1 = nn.Sequential(
            nn.Conv2d(in_dim, dim, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2)
        )
        self.l2 = nn.Sequential(*GAN_Block(dim, dim * 2, normalize=False))
        self.l3 = nn.Sequential(*GAN_Block(dim * 2, dim * 4, normalize=False))
        self.l4 = nn.Sequential(*GAN_Block(dim * 4, dim * 8, normalize=False))
        self.l5 = nn.Sequential(
            nn.Conv2d(dim * 8, 1, 4, 1, 0, bias=False)
        )
        self.apply(weights_init)

    def forward(self, x):
        return self.l5(self.l4(self.l3(self.l2(self.l1(x)))))

print("[INFO] 假畫大師 G 與 嚴厲鑑賞家 D 架構建立完成！")

In [ ]:
# =========================================================================
# 4. WGAN-GP 梯度懲罰核心邏輯 (Gradient Penalty)
# =========================================================================
def compute_gradient_penalty(D, real_samples, fake_samples, device):
    alpha = torch.rand((real_samples.size(0), 1, 1, 1), device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = D(interpolates)
    fake = torch.ones((real_samples.size(0), 1, 1, 1), device=device, requires_grad=False)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

print("[INFO] WGAN-GP 梯度懲罰防線部署完畢！")

In [ ]:
# =========================================================================
# 5. 訓練迴圈流程 (Training Pipeline)
# =========================================================================
# 超參數與基礎設定
batch_size = 64
z_dim = 100
lr = 1e-4
n_epoch = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 準備搬運車
dataset = get_dataset(os.path.join(workspace_dir, 'faces'))
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

# 實例化模型與教練
G = Generator(in_dim=z_dim).to(device)
D = Discriminator(in_dim=3).to(device)
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.9))
opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.9))
fixed_z = torch.randn(64, z_dim, 1, 1, device=device)

# 讀取中斷存檔機制
path_G_checkpoint = './G_checkpoint.pth'
path_D_checkpoint = './D_checkpoint.pth'
if os.path.exists(path_G_checkpoint) and os.path.exists(path_D_checkpoint):
    G.load_state_dict(torch.load(path_G_checkpoint, map_location=device))
    D.load_state_dict(torch.load(path_D_checkpoint, map_location=device))
    print(f"[LOAD] 🎉 成功偵測到延續檔案！已載入參數，繼續往下跑！")
else:
    print(f"[START] 沒找到存檔，正在使用 {device} 進行全新 WGAN-GP 訓練...")

# 開始決鬥大迴圈
for epoch in range(n_epoch):
    progress_bar = tqdm(dataloader, desc=f"Epoch [{epoch+1}/{n_epoch}]")
    for i, real_imgs in enumerate(progress_bar):
        real_imgs = real_imgs.to(device)

        # -------------------------------------------------
        # 戰場 1：練鑑賞家 D
        # -------------------------------------------------
        opt_D.zero_grad()
        z = torch.randn(batch_size, z_dim, 1, 1, device=device)
        fake_imgs = G(z).detach() # 砍斷 G 梯度

        loss_D = -torch.mean(D(real_imgs)) + torch.mean(D(fake_imgs))
        gp = compute_gradient_penalty(D, real_imgs, fake_imgs, device)
        loss_D += 10 * gp

        loss_D.backward()
        opt_D.step()

        # -------------------------------------------------
        # 戰場 2：練大師 G
        # -------------------------------------------------
        opt_G.zero_grad()
        z = torch.randn(batch_size, z_dim, 1, 1, device=device)
        gen_imgs = G(z)
        loss_G = -torch.mean(D(gen_imgs))

        loss_G.backward()
        opt_G.step()

        progress_bar.set_postfix(Loss_D=loss_D.item(), Loss_G=loss_G.item())

    # 🎬 【Notebook 獨家加碼：每回合完結立刻在網頁上秀出照片】
    G.eval()
    with torch.no_grad():
        generated_snapshot = G(fixed_z)
        # 先把成品存檔
        vutils.save_image(generated_snapshot, f"./epoch_{epoch+1}_output.png", normalize=True)

        # 核心畫圖動作：直接抓出來顯示在螢幕上
        clear_output(wait=True) # 擦掉上一輪的舊照片，保持畫面乾淨
        print(f"🔥 目前進度：第 {epoch+1} / {n_epoch} 回合成果圖")

        grid = vutils.make_grid(generated_snapshot, padding=2, normalize=True)
        np_grid = grid.cpu().numpy()
        plt.figure(figsize=(8, 8))
        plt.imshow(np.transpose(np_grid, (1, 2, 0))) # 換成 matplotlib 喜歡的 (H, W, C)
        plt.axis('off')
        plt.show() # 正式亮相！

    G.train()

    # 自動覆蓋存檔
    torch.save(G.state_dict(), path_G_checkpoint)
    torch.save(D.state_dict(), path_D_checkpoint)

torch.save(G.state_dict(), './generator.pth')
print("[SUCCESS] 50 回合圓滿結束！完全體大師權重已存入 generator.pth")